In [ ]:
import keras
from keras import layers, ops
import tensorflow as tf

LATENT_DIM = 128

class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = ops.shape(z_mean)[0]
        dim = ops.shape(z_mean)[1]
        epsilon = keras.random.normal(shape=(batch, dim), mean=0.0, stddev=1.0)
        return z_mean + ops.exp(0.5 * z_log_var) * epsilon

#Encoder
enc_in = layers.Input(shape=(64, 64, 3), name="encoder_input")

x = layers.Conv2D(64, kernel_size=4, strides=2, padding="same", use_bias=False)(enc_in)
x = layers.GroupNormalization(groups=8)(x)
x = layers.LeakyReLU(0.2)(x)

x = layers.Conv2D(128, kernel_size=4, strides=2, padding="same", use_bias=False)(x)
x = layers.GroupNormalization(groups=8)(x)
x = layers.LeakyReLU(0.2)(x)

x = layers.Conv2D(256, kernel_size=4, strides=2, padding="same", use_bias=False)(x)
x = layers.GroupNormalization(groups=8)(x)
x = layers.LeakyReLU(0.2)(x)

x = layers.Flatten()(x)
x = layers.Dense(512, activation="relu")(x)

z_mean = layers.Dense(LATENT_DIM, name="z_mean")(x)
z_log_var = layers.Dense(LATENT_DIM, name="z_log_var")(x)
z = Sampling()([z_mean, z_log_var])

encoder = keras.Model(enc_in, [z_mean, z_log_var, z], name="encoder")

#Decoder
dec_in = layers.Input(shape=(LATENT_DIM,), name="decoder_input")

x = layers.Dense(8 * 8 * 256, use_bias=False)(dec_in)
x = layers.Reshape((8, 8, 256))(x)
x = layers.GroupNormalization(groups=8)(x)
x = layers.LeakyReLU(0.2)(x)

x = layers.Conv2DTranspose(128, kernel_size=4, strides=2, padding="same", use_bias=False)(x)
x = layers.GroupNormalization(groups=8)(x)
x = layers.LeakyReLU(0.2)(x)

x = layers.Conv2DTranspose(64, kernel_size=4, strides=2, padding="same", use_bias=False)(x)
x = layers.GroupNormalization(groups=8)(x)
x = layers.LeakyReLU(0.2)(x)

dec_out = layers.Conv2DTranspose(3, kernel_size=4, strides=2, padding="same", activation="sigmoid")(x)

decoder = keras.Model(dec_in, dec_out, name="decoder")

#VAE
class VAE(keras.Model):
    def __init__(self, encoder, decoder, beta=0.001, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.beta = beta

        self.total_loss_tracker = keras.metrics.Mean(name="loss")
        self.rec_loss_tracker = keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.rec_loss_tracker,
            self.kl_loss_tracker,
        ]

    def call(self, inputs, training=False):
        z_mean, z_log_var, z = self.encoder(inputs, training=training)
        latent = z if training else z_mean
        return self.decoder(latent, training=training)

    @tf.function
    def train_step(self, data):
        x = data[0] if isinstance(data, (tuple, list)) else data

        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(x, training=True)
            reconstruction = self.decoder(z, training=True)

            x_tf = tf.convert_to_tensor(x)
            rec_tf = tf.convert_to_tensor(reconstruction)

            l1_loss = tf.reduce_mean(tf.abs(x_tf - rec_tf), axis=(1, 2, 3))
            ssim_val = tf.image.ssim(x_tf, rec_tf, max_val=1.0)
            ssim_loss = tf.reduce_mean(1.0 - ssim_val)
            rec_loss = (0.8 * ssim_loss + 0.2 * tf.reduce_mean(l1_loss)) * 1000.0

            kl_loss = -0.5 * tf.reduce_sum(1.0 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=-1)
            kl_loss = tf.reduce_mean(kl_loss)

            total_loss = rec_loss + (self.beta * kl_loss)

        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

        self.total_loss_tracker.update_state(total_loss)
        self.rec_loss_tracker.update_state(rec_loss)
        self.kl_loss_tracker.update_state(kl_loss)

        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.rec_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

    @tf.function
    def test_step(self, data):
        x = data[0] if isinstance(data, (tuple, list)) else data

        z_mean, z_log_var, z = self.encoder(x, training=False)
        reconstruction = self.decoder(z, training=False)

        x_tf = tf.convert_to_tensor(x)
        rec_tf = tf.convert_to_tensor(reconstruction)

        l1_loss = tf.reduce_mean(tf.abs(x_tf - rec_tf), axis=(1, 2, 3))
        ssim_val = tf.image.ssim(x_tf, rec_tf, max_val=1.0)
        ssim_loss = tf.reduce_mean(1.0 - ssim_val)
        rec_loss = (0.8 * ssim_loss + 0.2 * tf.reduce_mean(l1_loss)) * 1000.0

        kl_loss = -0.5 * tf.reduce_sum(1.0 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=-1)
        kl_loss = tf.reduce_mean(kl_loss)

        total_loss = rec_loss + (self.beta * kl_loss)

        self.total_loss_tracker.update_state(total_loss)
        self.rec_loss_tracker.update_state(rec_loss)
        self.kl_loss_tracker.update_state(kl_loss)

        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.rec_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

In [ ]:
vae = VAE(encoder, decoder, beta=0.001)
vae.compile(optimizer=keras.optimizers.Adam(learning_rate=2e-4))

In [ ]:
import os

TOKEN = ""

os.environ["KAGGLE_API_TOKEN"] = TOKEN

kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)
token_path = os.path.join(kaggle_dir, "access_token")

with open(token_path, "w") as f:
    f.write(TOKEN.strip())

os.chmod(token_path, 0o600)

!kaggle competitions download -c mg-practica-1-generando-caras
!unzip -o -q mg-practica-1-generando-caras.zip -d dataset_competencia
!ls -lh dataset_competencia

100% 186M/186M [00:01<00:00, 112MB/s]

total 8.0K
drwxr-xr-x 4 root root 4.0K Oct 24 12:59 CelebAMask
-rw-r--r-- 1 root root 3.3K Sep  8 09:49 Create_Submission.ipynb


In [ ]:
import tensorflow as tf

RUTA_FOTOS = "dataset_competencia/CelebAMask/images"

x_train = tf.keras.utils.image_dataset_from_directory(
    RUTA_FOTOS,
    labels=None,
    image_size=(64, 64),
    batch_size=128,
    validation_split=0.15,
    subset="training",
    seed=42,
    shuffle=True
)

x_test = tf.keras.utils.image_dataset_from_directory(
    RUTA_FOTOS,
    labels=None,
    image_size=(64, 64),
    batch_size=128,
    validation_split=0.15,
    subset="validation",
    seed=42,
    shuffle=False
)

x_train = x_train.map(lambda x: (x / 255.0, x / 255.0)).prefetch(tf.data.AUTOTUNE)
x_test = x_test.map(lambda x: (x / 255.0, x / 255.0)).prefetch(tf.data.AUTOTUNE)

Found 30000 files.
Using 25500 files for training.
Found 30000 files.
Using 4500 files for validation.


In [ ]:
vae_logs = vae.fit(
    x_train,
    epochs=50,
    validation_data=x_test,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5)
    ]
)

Epoch 1/50
 14/200 ━━━━━━━━━━━━━━━━━━━━ 14:47 5s/step - kl_loss: 847.9706 - loss: 774.3528 - reconstruction_loss: 773.5048

In [ ]:
vae_logs.history.keys()

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.plot(vae_logs.history['loss'])
plt.plot(vae_logs.history['val_loss'])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

images_batch, _ = next(iter(x_test))
original_imgs = images_batch.numpy()

z_mean, _, _ = vae.encoder.predict(images_batch)
decoded_imgs = vae.decoder.predict(z_mean)

n_images = 8
plt.figure(figsize=(16, 5))
for i in range(n_images):
    ax = plt.subplot(2, n_images, i + 1)
    plt.imshow(original_imgs[i])
    plt.title("Original")
    ax.axis("off")

    ax = plt.subplot(2, n_images, i + 1 + n_images)
    plt.imshow(decoded_imgs[i])
    plt.title("Reconstruida")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def slerp(val, low, high):
    low_norm = low / np.linalg.norm(low)
    high_norm = high / np.linalg.norm(high)
    omega = np.arccos(np.clip(np.dot(low_norm, high_norm), -1.0, 1.0))
    so = np.sin(omega)

    if so == 0:
        return (1.0 - val) * low + val * high

    scale_low = np.sin((1.0 - val) * omega) / so
    scale_high = np.sin(val * omega) / so

    interpolated = scale_low * low + scale_high * high
    return interpolated

img_A = images_batch[10]
img_B = images_batch[20]

z_mean, _, _ = vae.encoder.predict(np.array([img_A, img_B]))
z_A = z_mean[0]
z_B = z_mean[1]

z_50_linear = 0.5 * z_A + 0.5 * z_B

target_norm = (np.linalg.norm(z_A) + np.linalg.norm(z_B)) / 2.0
z_50_linear_norm = z_50_linear / np.linalg.norm(z_50_linear) * target_norm

z_50_slerp = slerp(0.5, z_A, z_B)

faces = vae.decoder.predict(np.array([z_A, z_50_linear, z_50_slerp, z_B]))

faces = np.clip(faces, 0.0, 1.0)
img_A_vis = np.clip(img_A, 0.0, 1.0)
img_B_vis = np.clip(img_B, 0.0, 1.0)

plt.figure(figsize=(15, 4))

plt.subplot(1, 5, 1)
plt.imshow(img_A_vis)
plt.title("Original A")
plt.axis("off")

plt.subplot(1, 5, 2)
plt.imshow(faces[0])
plt.title("Reconstrucción A")
plt.axis("off")

plt.subplot(1, 5, 3)
plt.imshow(faces[2])
plt.title("Mezcla Slerp 50/50\n(Más nítida)")
plt.axis("off")

plt.subplot(1, 5, 4)
plt.imshow(faces[3])
plt.title("Reconstrucción B")
plt.axis("off")

plt.subplot(1, 5, 5)
plt.imshow(img_B_vis)
plt.title("Original B")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import os
import numpy as np
import tensorflow as tf
from PIL import Image

n_images = 200
out_folder = "random_imgs"
os.makedirs(out_folder, exist_ok=True)


z_means = vae.encoder.predict(x_test, verbose=0)[0]
mu = np.mean(z_means, axis=0)
cov = np.cov(z_means, rowvar=False)

random_latent_vectors = np.random.multivariate_normal(mu, cov, size=n_images)

generated_imgs_rgb = vae.decoder.predict(random_latent_vectors, batch_size=64, verbose=0)
generated_imgs_rgb = np.clip(generated_imgs_rgb, 0.0, 1.0)

caras_bn = tf.image.rgb_to_grayscale(generated_imgs_rgb).numpy()

caras_bn_uint8 = (caras_bn.squeeze(-1) * 255.0).astype(np.uint8)

for i in range(n_images):
    img = Image.fromarray(caras_bn_uint8[i], mode="L")
    img.save(f"{out_folder}/img_{i+1:03d}.png")

print(f"Se han generado y guardado {n_images} imágenes en blanco y negro en '{out_folder}/'.")

In [ ]:
import glob, cv2, numpy as np, pandas as pd

paths = sorted(glob.glob("random_imgs/*.png"))
imgs_arr = [cv2.imread(p, cv2.IMREAD_GRAYSCALE).flatten() for p in paths]

df = pd.DataFrame(np.array(imgs_arr), columns=[str(i) for i in range(64*64)])
df.insert(0, "ID", range(1, len(df) + 1))
df.to_csv("submission.csv", index=False)

print(df.shape, df.columns[:4].tolist())